# ML-07 — Baseline Action Score and Top-20 Review

This notebook establishes a transparent, rule-based PRIORITY baseline for content review based on signal features.

## 1. My rule and its reason codes

**The Rule:** A page deserves review priority if it hasn't been updated in a while (stale, >= 91 days) but its target keyword has real search demand (high demand, >= 90th percentile) — representing an opportunity being left on the table.

**Reason Codes:**
* `STALE_HIGH_DEMAND`: Staleness >= 91 days AND Search Volume >= 100.
* `STALE_LOW_DEMAND`: Staleness >= 91 days AND Search Volume < 100.
* `FRESH_HIGH_DEMAND`: Staleness < 91 days AND Search Volume >= 100.
* `NO_KEYWORD_DATA`: Search Volume is null/NaN.
* `LOW_PRIORITY`: All other cases (e.g., fresh with low demand).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import pandas as pd
import numpy as np
import os

# 1. Load Data using relative path
input_path = 'my-ml-starter/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(input_path)

# 2. Data-grounded threshold for high_demand
quantiles = df['search_volume'].quantile([0.5, 0.75, 0.9])
print("Search Volume Quantiles:")
print(quantiles)

# Rule thresholds
STALE_THRESHOLD = 91
# We use 100 as instructed, which aligns with the 90th percentile
HIGH_DEMAND_THRESHOLD = 100

def assign_baseline(row):
    vol = row['search_volume']
    stale_days = row['days_since_last_update']

    if pd.isna(vol):
        return 0, "NO_KEYWORD_DATA"

    is_stale = stale_days >= STALE_THRESHOLD
    is_high_demand = vol >= HIGH_DEMAND_THRESHOLD

    # Scoring Logic
    score = int(is_stale) * int(is_high_demand) * vol

    # Reason Code Logic
    if is_stale and is_high_demand:
        code = "STALE_HIGH_DEMAND"
    elif is_stale and not is_high_demand:
        code = "STALE_LOW_DEMAND"
    elif not is_stale and is_high_demand:
        code = "FRESH_HIGH_DEMAND"
    else:
        code = "LOW_PRIORITY"

    return score, code

# 3. Apply Rule
results = df.apply(assign_baseline, axis=1)
df['score'], df['reason_codes'] = zip(*results)

# 4. Rank and Sort
df = df.sort_values(by='score', ascending=False)
df['rank'] = range(1, len(df) + 1)

# 5. Output
output_path = 'work/outputs/baseline_action_score.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)

print(f"\nProcessed {len(df)} rows. Saved to {output_path}.")
display(df[['rank', 'content_id', 'score', 'reason_codes', 'search_volume', 'days_since_last_update']].head(20))

Search Volume Quantiles:
0.50     10.0
0.75     20.0
0.90    110.0
Name: search_volume, dtype: float64

Processed 30000 rows. Saved to work/outputs/baseline_action_score.csv.


,rank,content_id,score,reason_codes,search_volume,days_since_last_update
12140,1,content_ef99c4abd9ab,74000.0,STALE_HIGH_DEMAND,74000.0,104
6972,2,content_bf67a444faef,60500.0,STALE_HIGH_DEMAND,60500.0,104
17907,3,content_5ec29ae79c60,60500.0,STALE_HIGH_DEMAND,60500.0,104
28282,4,content_454cc6654c6e,60500.0,STALE_HIGH_DEMAND,60500.0,104
18701,5,content_deb54e9e19cd,60500.0,STALE_HIGH_DEMAND,60500.0,104
15923,6,content_84fe9d0a707a,40500.0,STALE_HIGH_DEMAND,40500.0,104
2815,7,content_7868341d97dd,40500.0,STALE_HIGH_DEMAND,40500.0,104
3646,8,content_6b41450ae50c,27100.0,STALE_HIGH_DEMAND,27100.0,104
17859,9,content_5c5fab9d41e7,22200.0,STALE_HIGH_DEMAND,22200.0,104
3752,10,content_e7eb94e121b9,22200.0,STALE_HIGH_DEMAND,22200.0,104


## 3. Top-20 review

Reviewing the top picks with outcome context (avg_position, ctr) to see if the priority makes sense to a human reviewer.

In [4]:
# Load back from output path
ranked_df = pd.read_csv('work/outputs/baseline_action_score.csv')

# Outcome columns (avg_position, ctr) are for review context only, NOT scoring
review_cols = ['rank', 'content_id', 'score', 'reason_codes', 'avg_position', 'ctr']
display(ranked_df[review_cols].head(20))

,rank,content_id,score,reason_codes,avg_position,ctr
0,1,content_ef99c4abd9ab,74000.0,STALE_HIGH_DEMAND,38.5,0.03
1,2,content_bf67a444faef,60500.0,STALE_HIGH_DEMAND,45.5,0.00
2,3,content_5ec29ae79c60,60500.0,STALE_HIGH_DEMAND,49.8,0.00
3,4,content_454cc6654c6e,60500.0,STALE_HIGH_DEMAND,44.9,0.00
4,5,content_deb54e9e19cd,60500.0,STALE_HIGH_DEMAND,41.7,0.00
5,6,content_84fe9d0a707a,40500.0,STALE_HIGH_DEMAND,43.3,0.00
6,7,content_7868341d97dd,40500.0,STALE_HIGH_DEMAND,28.0,0.08
7,8,content_6b41450ae50c,27100.0,STALE_HIGH_DEMAND,43.2,0.00
8,9,content_5c5fab9d41e7,22200.0,STALE_HIGH_DEMAND,5.4,0.06
9,10,content_e7eb94e121b9,22200.0,STALE_HIGH_DEMAND,15.6,0.00


# 4. Weak picks + leakage check



In [6]:
filtered_df = ranked_df[ranked_df['avg_position'] > 0]

mean_top_50 = filtered_df.head(50)['avg_position'].mean()
mean_total = filtered_df['avg_position'].mean()

print(f"Mean Avg Position (Top 50): {mean_top_50:.2f}")
print(f"Mean Avg Position (Full Dataset): {mean_total:.2f}")

print("\nNOTE: This is a post-hoc directional check on the rule, not a trained evaluation. It is not proof of causation.")

Mean Avg Position (Top 50): 25.05
Mean Avg Position (Full Dataset): 17.03

NOTE: This is a post-hoc directional check on the rule, not a trained evaluation. It is not proof of causation.


### Hand-Review of Weak Picks

*   **Weak Pick Observation:** content_201a4a56f4d6 is ranked #14 with reason code STALE_HIGH_DEMAND (search_volume 22,200), but its avg_position is already 1.0 — the best possible position. The rule flags it as needing review purely because it's stale and high-demand, without accounting for the fact that it's already succeeding. This is a real limitation of the baseline: it can't see current performance, only staleness and demand.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.